# 05b — 진짜 하이브리드 모델 재학습
> **목표:** CNN(텍스처) + 픽셀 피처(Laplacian, FFT)를 feature-level concat으로 결합  
> **근거:** 픽셀 단독 Accuracy 50% → 단독 판정 불가 확인. CNN shared feature에 앵커로 concat  
> **파일명:** `05b_hybrid_retrain.ipynb`  
> **저장:** `stage3_hybrid_best.h5` + `pixel_scaler.pkl`

---
## 아키텍처 요약
```
이미지 입력(224,224,3)
    └─ MobileNetV2 백본(frozen) → GAP → Dense(256, relu) → Dropout(0.4)
                                                                    ↓
픽셀 피처 입력(2,) [Laplacian_norm, FFT_norm]                    Concat
    └─ Dense(32, relu) → Dense(32, relu)                           ↓
                                                        Dense(128, relu) → Dropout(0.3)
                                                            ↙           ↘
                                               head_binary(1, sigmoid)  head_spoof(4, softmax)
```
**손실:** BCE × 0.7 + CCE × 0.3  
**클래스 가중치:** {0: 3.0, 1: 1.0} (Live 과소 대응)  
**픽셀 정규화:** StandardScaler fit on train


## 0. 환경 설정 & Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE       = '/content/drive/MyDrive/face-anti-spoofing'
CROP_DIR   = f'{BASE}/data/cropped'
MODEL_DIR  = f'{BASE}/models'
REPORT_DIR = f'{BASE}/reports'

os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# GPU 확인
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
import numpy as np
import cv2
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 재현성
tf.random.set_seed(42)
np.random.seed(42)

print('패키지 로드 완료')

## 1. 픽셀 피처 추출 함수
> Laplacian 분산(선명도) + FFT 고주파 에너지 → 각 1개 스칼라 → 입력 벡터 크기 (2,)

In [ ]:
def extract_pixel_features(img_bgr: np.ndarray) -> np.ndarray:
    """
    입력: BGR 이미지 (224x224 또는 임의 크기)
    출력: [laplacian_var, fft_high_energy] shape=(2,)
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # --- 1) Laplacian 분산 (경계 선명도) ---
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    # --- 2) FFT 고주파 에너지 ---
    f = np.fft.fft2(gray.astype(np.float32))
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)
    h, w = magnitude.shape
    # 중심부(저주파) 제거: 전체의 1/4 반경
    cy, cx = h // 2, w // 2
    r = min(h, w) // 4
    mask = np.ones((h, w), np.uint8)
    cv2.circle(mask, (cx, cy), r, 0, -1)
    high_energy = (magnitude * mask).sum() / (magnitude.sum() + 1e-8)

    return np.array([lap_var, high_energy], dtype=np.float32)


# 빠른 동작 확인
dummy = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
feat  = extract_pixel_features(dummy)
print('픽셀 피처 shape:', feat.shape, '| 값:', feat)

## 2. 데이터셋 로드
> CROP_DIR 구조: `live/ print/ replay/ mask/` 각 1,500장  
> 라벨: binary (0=Live, 1=Spoof) / spoof_type (0=Live,1=Print,2=Replay,3=Mask)  
> **이 셀에서 이미지 + 픽셀 피처를 동시에 추출 → 메모리에 저장**

In [ ]:
LABEL_MAP = {'live': 0, 'print': 1, 'replay': 2, 'mask': 3}
IMG_SIZE  = (224, 224)

images        = []   # (N, 224, 224, 3)  float32 정규화
pixel_feats   = []   # (N, 2)            raw (나중에 StandardScaler)
labels_binary = []   # (N,)  0=Live, 1=Spoof
labels_spoof  = []   # (N,)  0~3

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

for cat, spoof_id in LABEL_MAP.items():
    cat_dir = Path(CROP_DIR) / cat
    files   = sorted(cat_dir.glob('*.jpg')) + sorted(cat_dir.glob('*.png'))
    print(f'  {cat}: {len(files)} 장 로드 중...')

    for fp in files:
        img_bgr = cv2.imread(str(fp))
        if img_bgr is None:
            continue

        # 픽셀 피처 추출 (resize 전 원본 크기에서)
        pf = extract_pixel_features(img_bgr)

        # 이미지 전처리
        img_resized = cv2.resize(img_bgr, IMG_SIZE)
        img_rgb     = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        img_norm    = (img_rgb.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD

        images.append(img_norm)
        pixel_feats.append(pf)
        labels_spoof.append(spoof_id)
        labels_binary.append(0 if cat == 'live' else 1)

images        = np.array(images,        dtype=np.float32)
pixel_feats   = np.array(pixel_feats,   dtype=np.float32)
labels_binary = np.array(labels_binary, dtype=np.float32)
labels_spoof  = np.array(labels_spoof,  dtype=np.int32)

print(f'\n총 샘플: {len(images)}')
print(f'이미지 shape: {images.shape}')
print(f'픽셀 피처 shape: {pixel_feats.shape}')
print(f'Binary 분포 — Live: {(labels_binary==0).sum()} / Spoof: {(labels_binary==1).sum()}')

## 3. Train / Val / Test 분할 + 픽셀 정규화

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42, stratify=labels_binary)
idx_val,   idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42,
                                        stratify=labels_binary[idx_tmp])

def split(arr, train, val, test):
    return arr[train], arr[val], arr[test]

X_img_tr,  X_img_va,  X_img_te  = split(images,        idx_train, idx_val, idx_test)
X_pix_tr,  X_pix_va,  X_pix_te  = split(pixel_feats,   idx_train, idx_val, idx_test)
y_bin_tr,  y_bin_va,  y_bin_te  = split(labels_binary,  idx_train, idx_val, idx_test)
y_sp_tr,   y_sp_va,   y_sp_te   = split(labels_spoof,   idx_train, idx_val, idx_test)

# StandardScaler — train fit, val/test transform
scaler = StandardScaler()
X_pix_tr = scaler.fit_transform(X_pix_tr)
X_pix_va = scaler.transform(X_pix_va)
X_pix_te = scaler.transform(X_pix_te)

# Scaler 저장 (추론 시 재사용)
with open(f'{MODEL_DIR}/pixel_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'Train: {len(X_img_tr)} | Val: {len(X_img_va)} | Test: {len(X_img_te)}')
print(f'픽셀 피처 — Train mean: {X_pix_tr.mean(axis=0).round(4)} std: {X_pix_tr.std(axis=0).round(4)}')
print('Scaler 저장 완료: pixel_scaler.pkl')

## 4. 하이브리드 모델 정의
```
img_input → MobileNetV2(frozen) → GAP → Dense(256) → Dropout(0.4)
                                                              ↓
pix_input → Dense(32) → Dense(32)                         Concat
                                                              ↓
                                              Dense(128) → Dropout(0.3)
                                              ↙                      ↘
                            head_binary(1, sigmoid)      head_spoof(4, softmax)
```

In [ ]:
def build_hybrid_model(pixel_dim: int = 2) -> Model:
    # --- 이미지 경로 ---
    img_input  = layers.Input(shape=(224, 224, 3), name='img_input')
    backbone   = MobileNetV2(include_top=False, weights='imagenet',
                              input_tensor=img_input)
    backbone.trainable = False   # Phase 1: backbone frozen

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='img_dense')(x)
    x = layers.Dropout(0.4, name='img_drop')(x)

    # --- 픽셀 경로 ---
    pix_input = layers.Input(shape=(pixel_dim,), name='pix_input')
    p = layers.Dense(32, activation='relu', name='pix_dense1')(pix_input)
    p = layers.Dense(32, activation='relu', name='pix_dense2')(p)

    # --- 결합 ---
    merged = layers.Concatenate(name='concat')([x, p])
    merged = layers.Dense(128, activation='relu', name='shared_dense')(merged)
    merged = layers.Dropout(0.3, name='shared_drop')(merged)

    # --- 출력 헤드 ---
    head_binary = layers.Dense(1, activation='sigmoid', name='head_binary')(merged)
    head_spoof  = layers.Dense(4, activation='softmax', name='head_spoof')(merged)

    model = Model(
        inputs  = [img_input, pix_input],
        outputs = [head_binary, head_spoof],
        name    = 'hybrid_fas'
    )
    return model


model = build_hybrid_model(pixel_dim=2)
model.summary(line_length=90)

## 5. 컴파일 & 클래스 가중치

In [ ]:
# 손실 가중치: binary 중심 (0.7) + spoof type 보조 (0.3)
model.compile(
    optimizer = optimizers.Adam(learning_rate=1e-3),
    loss = {
        'head_binary': 'binary_crossentropy',
        'head_spoof' : 'sparse_categorical_crossentropy'
    },
    loss_weights = {'head_binary': 0.7, 'head_spoof': 0.3},
    metrics = {
        'head_binary': ['accuracy'],
        'head_spoof' : ['accuracy']
    }
)

# class_weight는 멀티 출력 모델에서 미지원 → sample_weight로 대체
# Live(0) 과소 보정 — Live 1500 vs Spoof 4500 → 비율 3:1
def make_sample_weight(y_binary: np.ndarray,
                        live_w: float = 3.0,
                        spoof_w: float = 1.0) -> np.ndarray:
    """이진 라벨(0=Live, 1=Spoof)에 따라 샘플별 가중치 배열 반환"""
    w = np.where(y_binary == 0, live_w, spoof_w).astype(np.float32)
    return w

sw_train = make_sample_weight(y_bin_tr)
sw_val   = make_sample_weight(y_bin_va)
print(f'sample_weight — Live(3.0): {(sw_train==3).sum()} / Spoof(1.0): {(sw_train==1).sum()}')
print('컴파일 완료')

## 6. Phase 1 — Backbone Frozen 학습 (Top Layer만)

In [ ]:
CKPT_PATH = f'{MODEL_DIR}/stage3_hybrid_best.h5'

cb_list = [
    callbacks.ModelCheckpoint(
        CKPT_PATH,
        monitor='val_head_binary_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_head_binary_accuracy',
        patience=5,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print('=== Phase 1: Backbone Frozen ===')
hist1 = model.fit(
    x = [X_img_tr, X_pix_tr],
    y = {'head_binary': y_bin_tr, 'head_spoof': y_sp_tr},
    validation_data = (
        [X_img_va, X_pix_va],
        {'head_binary': y_bin_va, 'head_spoof': y_sp_va},
        sw_val
    ),
    epochs          = 20,
    batch_size      = 32,
    sample_weight   = sw_train,
    callbacks       = cb_list,
    verbose         = 1
)

## 7. Phase 2 — Backbone 일부 Unfreeze (Fine-tuning)

In [ ]:
# MobileNetV2 마지막 30개 레이어만 unfreeze
backbone_layer = model.get_layer('mobilenetv2_1.00_224')
backbone_layer.trainable = True

# 상위 30개만 학습, 나머지 frozen
for layer in backbone_layer.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f'학습 가능 레이어: {trainable_count} / {len(model.layers)}')

# 낮은 LR로 재컴파일
model.compile(
    optimizer = optimizers.Adam(learning_rate=1e-4),
    loss = {
        'head_binary': 'binary_crossentropy',
        'head_spoof' : 'sparse_categorical_crossentropy'
    },
    loss_weights = {'head_binary': 0.7, 'head_spoof': 0.3},
    metrics = {
        'head_binary': ['accuracy'],
        'head_spoof' : ['accuracy']
    }
)

print('\n=== Phase 2: Fine-tuning (LR=1e-4) ===')
hist2 = model.fit(
    x = [X_img_tr, X_pix_tr],
    y = {'head_binary': y_bin_tr, 'head_spoof': y_sp_tr},
    validation_data = (
        [X_img_va, X_pix_va],
        {'head_binary': y_bin_va, 'head_spoof': y_sp_va},
        sw_val
    ),
    epochs          = 20,
    batch_size      = 32,
    sample_weight   = sw_train,
    callbacks       = cb_list,
    verbose         = 1
)

## 8. 학습 곡선 시각화

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for k in h1.history:
        merged[k] = h1.history[k] + h2.history.get(k, [])
    return merged

hist = merge_histories(hist1, hist2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Binary Accuracy
axes[0].plot(hist['head_binary_accuracy'],     label='Train Binary Acc')
axes[0].plot(hist['val_head_binary_accuracy'], label='Val Binary Acc')
axes[0].axvline(len(hist1.history['head_binary_accuracy']) - 1,
                color='gray', linestyle='--', label='Fine-tune 시작')
axes[0].set_title('Binary Accuracy')
axes[0].legend()
axes[0].set_ylim(0.5, 1.01)

# Spoof Type Accuracy
axes[1].plot(hist['head_spoof_accuracy'],     label='Train Spoof Acc')
axes[1].plot(hist['val_head_spoof_accuracy'], label='Val Spoof Acc')
axes[1].axvline(len(hist1.history['head_spoof_accuracy']) - 1,
                color='gray', linestyle='--')
axes[1].set_title('Spoof Type Accuracy')
axes[1].legend()
axes[1].set_ylim(0.5, 1.01)

# Total Loss
axes[2].plot(hist['loss'],     label='Train Loss')
axes[2].plot(hist['val_loss'], label='Val Loss')
axes[2].axvline(len(hist1.history['loss']) - 1, color='gray', linestyle='--')
axes[2].set_title('Total Loss')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/05b_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('학습 곡선 저장 완료')

## 9. 전체 성능 평가 (Test Set)

In [ ]:
# 최적 모델 로드
model.load_weights(CKPT_PATH)

pred_bin_prob, pred_sp_prob = model.predict(
    [X_img_te, X_pix_te], batch_size=64, verbose=1
)
pred_bin = (pred_bin_prob.squeeze() >= 0.5).astype(int)
pred_sp  = pred_sp_prob.argmax(axis=1)

# ROC-AUC
auc = roc_auc_score(y_bin_te, pred_bin_prob.squeeze())

print('\n===== 전체 성능 =====')
print(f'Binary Accuracy : {(pred_bin == y_bin_te).mean():.4f}')
print(f'ROC-AUC         : {auc:.4f}')
print()
print('Classification Report (Binary):')
print(classification_report(y_bin_te, pred_bin, target_names=['Live', 'Spoof']))

## 10. 공격 유형별 FAR (핵심 실험)

In [ ]:
ATTACK_NAMES = {1: 'Print Attack', 2: 'Replay Attack', 3: '3D Mask Attack'}
FAR_TARGETS  = {1: 5.0, 2: 10.0, 3: 8.0}   # % 목표

far_results = {}

print('===== 공격 유형별 FAR 분석 =====')
print(f'{"공격 유형":<18} {"샘플":>6} {"FAR":>8} {"목표":>8} {"달성":>6}')
print('-' * 55)

for attack_id, attack_name in ATTACK_NAMES.items():
    # 해당 공격 유형 인덱스 추출
    mask = (y_sp_te == attack_id)
    if mask.sum() == 0:
        print(f'{attack_name:<18} — 샘플 없음')
        continue

    n_attack   = mask.sum()
    n_pass     = (pred_bin[mask] == 0).sum()   # Spoof를 Live로 오판 = FAR
    far_pct    = n_pass / n_attack * 100
    target     = FAR_TARGETS[attack_id]
    achieved   = '✅' if far_pct <= target else '❌'

    far_results[attack_name] = {
        'n_samples': int(n_attack),
        'n_pass'   : int(n_pass),
        'FAR_pct'  : round(far_pct, 2),
        'target_pct': target,
        'achieved' : far_pct <= target
    }

    print(f'{attack_name:<18} {n_attack:>6} {far_pct:>7.2f}% {target:>7.1f}% {achieved:>6}')

# JSON 저장
with open(f'{REPORT_DIR}/05b_far_results.json', 'w', encoding='utf-8') as f:
    json.dump(far_results, f, ensure_ascii=False, indent=2)

print('\nFAR 결과 저장 완료: 05b_far_results.json')

## 11. FAR 시각화 (막대 그래프)

In [ ]:
if far_results:
    names    = list(far_results.keys())
    far_vals = [far_results[n]['FAR_pct']  for n in names]
    targets  = [far_results[n]['target_pct'] for n in names]
    colors   = ['#2ecc71' if far_results[n]['achieved'] else '#e74c3c' for n in names]

    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(x - width/2, far_vals, width, label='실측 FAR', color=colors, alpha=0.85)
    ax.bar(x + width/2, targets, width, label='목표 FAR', color='#3498db', alpha=0.5)

    for bar, val in zip(bars, far_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

    ax.set_xlabel('공격 유형', fontsize=13)
    ax.set_ylabel('FAR (%)', fontsize=13)
    ax.set_title('공격 유형별 FAR — 하이브리드 모델', fontsize=15)
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=12)
    ax.legend(fontsize=12)
    ax.set_ylim(0, max(max(far_vals), max(targets)) * 1.3 + 2)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/05b_far_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('FAR 막대 그래프 저장 완료')

## 12. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary CM
cm_bin = confusion_matrix(y_bin_te, pred_bin)
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Live', 'Spoof'], yticklabels=['Live', 'Spoof'])
axes[0].set_title('Binary Confusion Matrix', fontsize=13)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Spoof Type CM
type_labels = ['Live', 'Print', 'Replay', 'Mask']
cm_sp = confusion_matrix(y_sp_te, pred_sp)
sns.heatmap(cm_sp, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=type_labels, yticklabels=type_labels)
axes[1].set_title('Spoof Type Confusion Matrix', fontsize=13)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/05b_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion Matrix 저장 완료')

## 13. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_bin_te, pred_bin_prob.squeeze())

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2980b9', lw=2, label=f'ROC (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random')
plt.xlabel('False Positive Rate (FAR)', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — 하이브리드 모델', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/05b_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('ROC Curve 저장 완료')

## 14. 요약 리포트 출력 & GitHub 커밋 가이드

In [ ]:
print('=' * 60)
print('  05b 하이브리드 모델 — 최종 결과 요약')
print('=' * 60)
binary_acc = (pred_bin == y_bin_te).mean()
print(f'  Binary Accuracy : {binary_acc:.4f} ({binary_acc*100:.2f}%)')
print(f'  ROC-AUC         : {auc:.4f}')
print()
print('  공격 유형별 FAR:')
for name, r in far_results.items():
    status = '✅ 목표 달성' if r['achieved'] else '❌ 목표 미달'
    print(f'    {name:<20}: {r["FAR_pct"]:>5.1f}% (목표 {r["target_pct"]}%) {status}')
print()
print('  저장 파일:')
print(f'    모델   : {CKPT_PATH}')
print(f'    스케일러: {MODEL_DIR}/pixel_scaler.pkl')
print(f'    리포트 : {REPORT_DIR}/05b_*.png, 05b_far_results.json')
print('=' * 60)
print()
print('>>> GitHub 커밋 가이드:')
print('  git add notebooks/05b_hybrid_retrain.ipynb')
print('  git add results/week13/')
print('  git commit -m "feat: hybrid CNN+pixel model stage3 training"')
print('  git commit -m "results: FAR analysis print/replay/mask"')

---
## ⏭️ 다음 단계 (06_gradcam.ipynb)

1. `stage3_hybrid_best.h5` 로드
2. `pixel_scaler.pkl` 로드
3. GradientTape 기반 Grad-CAM 구현 (`head_binary` 기준)
4. Real / Fake 샘플 각 10장씩 히트맵 시각화
5. 공격 유형별 히트맵 패턴 비교 (Print vs Replay vs Mask)

### 아키텍처 메모 (XAI 구현 시 참고)
```python
# Grad-CAM 대상 레이어 이름
target_layer = 'mobilenetv2_1.00_224'  # 마지막 conv 블록
# 픽셀 입력은 고정값으로 전달
inputs = [img_batch, pix_batch]
```